# Добавить пользователей с полным доступом (ГО ДТПП)

Роль `GO_DTPP` + `is_all_filials = '1'`: все листы, все филиалы. Добавляет только строки в `sbx_da.rls_acq_user`; если логин уже есть, его не трогает. Таблицы не пересоздаёт (в отличие от Run All в `rls_acq_roles_build`).

Логин должен совпадать с RAISA: `SELECT '{{ current_username() }}'` в SQL Lab под этим человеком. Регистр не важен.

Запускать сверху вниз под своей учёткой DRP. Если LDAP отказал, не повторяйте больше 2 раз.

In [ ]:
from getpass import getpass

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

NEW_USERS = [
    "safonov-an",
    "gorelov-aalek",
    "grishina-nn",
    "nikolaeva-yuan",
]
ROLE = "GO_DTPP"
COMMENT = "ГО, полный доступ: все листы, все РФ"
DRP_USER_DEFAULT = "Shestopalov-VYur"

if "drp" not in globals() or drp is None:
    drp_user = input("DRP user: ").strip() or DRP_USER_DEFAULT
    drp = connect(to="DRP", user_params={"user_name": drp_user, "password": getpass("DRP password: ")})


def fetch(sql):
    with drp:
        df = drp.fetch(sql)
    return pd.DataFrame() if df is None else df


def q(s):
    return str(s).strip().replace("'", "''")


users_in = ", ".join(f"lower('{q(u)}')" for u in NEW_USERS)
USERS_SQL = (
    "SELECT username, role, is_all_filials, comment FROM sbx_da.rls_acq_user "
    f"WHERE lower(BTRIM(username)) IN ({users_in}) ORDER BY 1"
)

display(fetch("SELECT current_user AS me"))
print("Уже есть в справочнике:")
display(fetch(USERS_SQL))

## Добавление

Повторный запуск безопасен: существующие логины пропускаются.

In [ ]:
with drp:
    drp.execute("SET statement_timeout = '30s'")
    for u in NEW_USERS:
        drp.execute(
            "INSERT INTO sbx_da.rls_acq_user (username, role, is_all_filials, comment) "
            f"SELECT '{q(u)}', '{ROLE}', '1', '{q(COMMENT)}' "
            "WHERE NOT EXISTS (SELECT 1 FROM sbx_da.rls_acq_user "
            f"WHERE lower(BTRIM(username)) = lower('{q(u)}'))"
        )
        print("OK:", u)

result = fetch(USERS_SQL)
display(result)

found = set(result["username"].str.strip().str.lower()) if not result.empty else set()
missing = [u for u in NEW_USERS if u.lower() not in found]
wrong = result[(result["role"].str.strip() != ROLE) | (result["is_all_filials"].astype(str).str.strip() != "1")] if not result.empty else result

if missing:
    print("НЕ ДОБАВЛЕНЫ:", missing)
elif not wrong.empty:
    print("Эти логины уже были с другой ролью, тетрадка их не меняла:")
    display(wrong)
else:
    print("ГОТОВО: все", len(NEW_USERS), "с ролью", ROLE, "и всеми филиалами")